In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
features = ["engine_displacement", "horsepower", "vehicle_weight", "model_year"]
target = "fuel_efficiency_mpg"
df = pd.read_csv(url)[features + [target]]

def split(seed):
    idx = np.arange(len(df))
    np.random.seed(seed)
    np.random.shuffle(idx)
    n_val = n_test = int(len(df) * 0.2)
    n_train = len(df) - n_val - n_test
    return (df.iloc[idx[:n_train]],
            df.iloc[idx[n_train:n_train + n_val]],
            df.iloc[idx[n_train + n_val:]])

def rmse(train, valid, fill=0, r=0):
    X = np.column_stack([np.ones(len(train)), train[features].fillna(fill)])
    Xv = np.column_stack([np.ones(len(valid)), valid[features].fillna(fill)])
    y, yv = train[target].to_numpy(), valid[target].to_numpy()
    w = np.linalg.solve(X.T @ X + r * np.eye(X.shape[1]), X.T @ y)
    return np.sqrt(np.mean((yv - Xv @ w) ** 2))

train, valid, test = split(42)
print("Q1:", df[features].isna().sum().idxmax())
print("Q2:", df.horsepower.median())
print("Q3 (zero, mean):", round(rmse(train, valid), 3),
      round(rmse(train, valid, train.horsepower.mean()), 3))

rates = [0, 0.01, 0.1, 1, 5, 10, 100]
scores = {r: round(rmse(train, valid, r=r), 4) for r in rates}
print("Q4:", scores, "best =", min(scores, key=scores.get))

seed_scores = [rmse(split(seed)[0], split(seed)[1]) for seed in range(10)]
print("Q5:", round(np.std(seed_scores), 3))

train, valid, test = split(9)
print("Q6:", round(rmse(pd.concat([train, valid]), test, r=0.001), 3))

Q1: horsepower
Q2: 254.0
Q3 (zero, mean): 2.205 2.202
Q4: {0: np.float64(2.2053), 0.01: np.float64(2.2058), 0.1: np.float64(2.2241), 1: np.float64(2.3492), 5: np.float64(2.4094), 10: np.float64(2.4195), 100: np.float64(2.4292)} best = 0
Q5: 0.029
Q6: 2.236
